<a href="https://colab.research.google.com/github/galaktrida/project-Nabokov/blob/main/%D0%9F%D0%A0%D0%9E%D0%95%D0%9A%D0%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Анализ корпуса рассказов Леонида Андреева
В качестве базы для нашего исследования мы выбрали рассказы Леонида Андреева.

За основу корпуса был взят сборник с сайта https://royallib.com/book/andreev_leonid/sbornik_rasskazov.html?ysclid=mmj1os7ar8411492521, куда вручную добавлены недостающие рассказы с сайта http://az.lib.ru/a/andreew_l_n/?ysclid=mmj1o5nusf5741485.

#Цели проекта
Необходимо создать программу, которая поможет получить из корпуса текстов данные, на основе которых можно исследовать творчество Л. Андреева. Его тексты считаются мрачными и наполненными поэтикой безумия, страха, угнетённости и неблагополучия, а синтаксис описательным и запутывающим. Созданная программа позволит подтвердить или опровергнуть некоторые из этих гипотез, также она может быть использована для дальнейших исследований.

**Исследоватлеьский вопрос:** Что компьютерный анализ может сказать о художественном стиле Л. Андреева? Как объективные данные соотносятся с существующими стереотипами и принятыми в литературоведческом сообществе гипотезами?

#Задачи

1. Предварительно обработать текстовый файл
2. Создать словарь для работы с рассказами как с отдельными элементами
3. Определить удобочитаемость текста
4. Проанализировать среднюю длину предложения для каждого рассказа
5. Выявить частотные леммы, биграммы и триграммы для каждого рассказа
6. Создать механизм поиска конкордансов для выбранного слова
7. Выявить ключевые слова для каждого рассказа
8. Проанализировать долю частей речи во всём корпусе

#Предобработка
В текстовом файле рассказы разделены маркером ===STORY===, а после названий поставлены точки, чтобы их можно было считывать универсально как первое предложение после маркера и, следовательно, использовать как название элемента.

In [ ]:
!pip install pymorphy3
import re
import pandas as pd
from nltk import download
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from pymorphy3 import MorphAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.probability import FreqDist
from nltk import bigrams
from nltk import trigrams
from nltk import Text
from functools import lru_cache
download('punkt_tab')
download('stopwords')

morph = MorphAnalyzer()
cache = {}
def fast_lemma(word): #функция для кэширования, чтобы код работал быстрее
  if word in cache:
    return cache[word]
  parsed = morph.parse(word)[0]
  lemma = parsed.normal_form
  cache[word] = lemma
  return lemma


input_file = "/content/Андреев. Рассказы. Сборник.txt"
marker = "===STORY==="
text = ''
dictionary = {} #создаём словарь для соотношения названий рассказов и их текстов

with open(input_file, 'r', encoding="utf-8") as f: #открываем файл для чтения
    for line in f:
      text += line.strip().replace('\xa0', ' ') #записываем в переменную text строки без символов переносов строки


corpus = text.split(marker) #разделяем рассказы по маркеру
del corpus[0] #при сплите по маркеру первый элемент нового списка - пустая строка, убираем
corpus_sent = [] #создаём список текстов, где они разбиты по предложениям

for story in corpus:
  sent = re.split(r'[.!?]+', story) #используем библиотеку re. тут создаётся список, разбитый по ., ! и ? знаку
  if sent[-1] != '': #иногда при разбиении последняя переменная - пустая строка
    corpus_sent.append(sent)
  else:
    del sent[-1] #если последняя переменная - пустая строка, мы её убираем
    corpus_sent.append(sent)

for count in range(len(corpus_sent)):
  dictionary[corpus_sent[count][0]] = corpus_sent[count][1:] #делаем словарь, где ключ - название рассказа, а значение - список всех его предложений

for element in range(len(corpus_sent)):
  del corpus_sent[element][0] #убираем из списка с предложениями названия рассказов - мы их не считаем

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


FileNotFoundError: [Errno 2] No such file or directory: '/content/Андреев. Рассказы. Сборник.txt'

#Определение удобочитаемости
Этот анализ позволяет определить, насколько текст «дружелюбен» к читателю.

In [ ]:
import re
import math

#для определения числа слогов будем считать количество гласных в слове.

def count_syllables_ru(word):
    vowels = "ауоыиэяюеё"
    count = 0
    for char in word.lower():
        if char in vowels:
            count += 1
    return max(1, count) #если в слове нет гласных, считаем 1 слог


def calculate_flesch_reading_ease_ru(text):

    words = text.split()
    num_words = len(words)

    #подсчет предложений, заканчиваются на ., !, ?
    sentences = re.split(r'[.!?]+', text)
    #отфильтруем пустые строки, которые могут появиться после сплита
    sentences = [s for s in sentences if s.strip()]
    num_sentences = len(sentences)

    #подсчет слогов
    num_syllables = 0
    for word in words:
        num_syllables += count_syllables_ru(word)

    #предотвращение деления на ноль
    if num_words == 0 or num_sentences == 0:
        return 0.0

    #оригинальная формула: 206.835 - 1.015 * (average_sentence_length) - 84.6 * (average_syllables_per_word)
    #адаптируем коэффициенты для русского языка
    average_sentence_length = num_words / num_sentences
    average_syllables_per_word = num_syllables / num_words

    #пример адаптированной формулы (коэффициенты подобраны эвристически)
    flesch_score = 206.835 - (1.3 * average_sentence_length) - (60.1 * average_syllables_per_word)

    return flesch_score


count = 0
for i in corpus:
  flesch_score_ru = calculate_flesch_reading_ease_ru(i)
  print(f"Индекс удобочитаемости Флеша рассказа {list(dictionary.keys())[count]} {flesch_score_ru:.2f}")
  count += 1


#Расчёт средней длины предложеиня для каждого рассказа

Анализ средней длины предложения позволяет оценить ритм и характерный для Л. Андреева синтаксис. При соотнеснении темы и показателя средней длины предложения можно подтвердить или опровергнуть гипотезу о связи ритма с эмоциональной окраской или темой произведения.
Этот анализ проводим до внедрения стоп-слов.

In [ ]:
dict_sentences = {} #считаем среднюю длину предложения
count = 0
for story in corpus_sent:
  n_sent = len(story) #считаем количество предложений
  lengths = [] #список для длин каждого предложения
  for sent in story:
    words = sent.split()
    lengths.append(len(words))
  summ_sent = sum(lengths) / n_sent
  dict_sentences[list(dictionary.keys())[count]] = round(summ_sent, 3) #округление до 3 знака после запятой
  count += 1
print(dict(sorted(dict_sentences.items(), key=lambda x: x[1], reverse=True))) #отсортированный словарь по убыванию значений

#Поиск частотных лемм, биграмм и триграмм
Для выполнения этой задачи провродим лемматизацию, токенизацию и очистку текста.
Частотный лексический анализ позволяет выявить основные художественные образы и устойчивые лексические конструкции. Такой анализ помогает охарактеризовать общую образность Л. Андреева и проследить, как его языковая палитра меняется в зависимости от темы произведения.

In [ ]:
for story in corpus_sent: #проводим лемматизацию по предложениям
  for sent in range(len(story)):
    text_lower = story[sent].lower()
    text_clean = re.sub(r'[/^/&/*/%]', '', text_lower) # sub заменяет что-то на что-то, убираем дополнительные ненужные знаки
    text_clean = re.sub(r'[^а-яё\s]', '', text_lower) # оставляем только русские символы
    text_clean = re.sub(r'\s+', ' ', text_clean).strip() # заменяем множественные пробелы на один
    story[sent] = text_clean # заменяем предложение на "очищенное"

morph = MorphAnalyzer() #создаём объект, который поможет привести слово к начальной форме
# очистка и токенизация
custom_stops = []
with open('/content/стоп-слова.txt', 'r', encoding='utf-8') as f: #загружаем собственные стоп-слова
    for line in f:
      word = line.strip()
      if word:
        lemma = fast_lemma(word) #добавляем именно лемму, чтобы не сравнивать с многими формами
        custom_stops.append(lemma)
stop_words = set(stopwords.words('russian') + custom_stops) #превращаем в множество, чтобы не было повторяющихся стоп-слов

parts_cleaned = [] # список токенов для каждой главы
for part in corpus_sent:
  sent_tokens = [] # список токенов для каждого предложения. при новом "проходе" через цикл обнуляется, что позволяет не дублировать токены в основной список parts_cleaned
  for sent in part:
    tokens = word_tokenize(sent) # токенизация с помощью NLTK
    lemmas_part = []
    for token in tokens:
      token_lemma = fast_lemma(token) # приводим слово к начальной форме
      lemmas_part.append(token_lemma)
    filtered_tokens = [token for token in lemmas_part if token.isalnum() and token not in stop_words] # удаление стоп-слов
    sent_tokens.append(' '.join(filtered_tokens)) # "соединяем" список токенов в предложении по пробелу
  parts_cleaned.append(' '.join(sent_tokens))

In [ ]:
#для каждого рассказа выводим 10 самых частых лемм
count = 0 #счётчик для названия рассказа из словаря
for story in parts_cleaned:
  word_frequencies = FreqDist(story.split()) # создание частотного словаря
  print(f"10 самых частых лемм рассказа {list(dictionary.keys())[count]} {list(word_frequencies.most_common(10))}")
  bigram_list = list(bigrams(story.split())) # при преобразовании в список даёт кортежи вида (слово1, слово2)
  fdist_bigrams = FreqDist(bigram_list)
  print("Топ-10 самых частотных биграмм:") #выводим самые частотные биграммы для каждого рассказа
  for bg, freq in fdist_bigrams.most_common(10):
    print(f"{bg[0]} {bg[1]}: {freq}")
  trigram_list = list(trigrams(story.split()))
  fdist_trigrams = FreqDist(trigram_list)
  print("Топ-10 самых частотных триграмм:")
  for tg, freq in fdist_trigrams.most_common(10): #выводим самые частотные триграммы для каждого рассказа
    print(f"{tg[0]} {tg[1]} {tg[2]}: {freq}")
  count += 1

#Поиск конкордансов
Поиск конкордансов позволяет узнать словоупотребительный контекст для любого слова. В перспективе исследования можно сравнить контекст слов в корпусе Л. Андреева и в НКРЯ.


In [ ]:
count = 0 #счётчик для названия рассказа из словаря
search_word = 'крик'
for story in parts_cleaned: #делаем конкорданс
  corpus_nltk = Text(story.split())
  if search_word in corpus_nltk:
    print(f'Конкорданс для рассказа {list(dictionary.keys())[count]}')
    corpus_nltk.concordance(search_word, lines=15, width=80)
  count += 1

#Поиск ключевых слов для каждого рассказа
Набор ключевых слов демонстриует основные темы, образы и мотивы каждого рассказа. При сопоставлении наборов ключевых слов можно сразу выделить уникальное настроение и\или тематику конкретного произведения.

In [ ]:
make_tf_idf = TfidfVectorizer() # создаём специальный объект-векторайзер
tf_idf = make_tf_idf.fit_transform(parts_cleaned) # кладем в этот векторайзер наш список и просим сделать табличку с расчетом TF_IDF
data = pd.DataFrame(tf_idf.todense()) # создаём полную матрицу тексты-слова
data.columns = make_tf_idf.get_feature_names_out() # называем колонки по словам
data_text_as_columns = data.T
print(data_text_as_columns)

In [ ]:
top_n = 20
# словарь {рассказ: [ключевые слова]}
keywords_per_doc = {}
for col in data_text_as_columns.columns:
    # сортируем слова по весу tf-idf
    top_words = (
        data_text_as_columns[col]
        .sort_values(ascending=False)
        .head(top_n)
        .index.tolist()
    )
    keywords_per_doc[col] = top_words

for doc, words in keywords_per_doc.items():
    print(f"Рассказ {list(dictionary.keys())[doc]}: {', '.join(words)}")

#Визуализация частотных лемм
Строим облако слов на освнове частотности употребления леммы во всём корпусе.

In [ ]:
# преобразуем список лемм в единый текст (разделённый пробелами)
text_for_cloud = ' '.join(parts_cleaned)

# создаём объект WordCloud и генерируем изображение
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis',  # цветовая схема
    max_words=100,       # максимум слов
    random_state=42      # для воспроизводимости
).generate(text_for_cloud) # анализирует текст, подсчитывает частоты слов и создаёт изображение, где размер слова пропорционален частоте

# отрисовываем
plt.figure(figsize=(12, 6)) # задаём размер картинки
plt.imshow(wordcloud)
plt.axis('off')  # убираем оси
plt.title('Облако наиболее частотных лемм в рассказах Андреева', fontsize=16)
plt.show()

#Анализ доли существительных, прилагательных и глаголов
Такой анализ позволяет понять, что в тексте доминирует: описания, объекты или действия.

In [ ]:
# считаем долю глаголов, прилагательных и существительных в текстах
n_words = 0
n_verb = 0
n_adj = 0
n_noun = 0
for story in corpus_sent:
  for sent in story:
    words = sent.split()
    n_words += len(words)
    for word in words:
      parsed = morph.parse(word)[0]
      pos = parsed.tag.POS
      if pos == 'VERB':
        n_verb += 1
      elif pos == 'ADJF' or pos == 'ADJS':
        n_adj += 1
      elif pos == 'NOUN':
        n_noun += 1
print(round(n_verb / n_words, 3))
print(round(n_adj / n_words, 3))
print(round(n_noun / n_words, 3))

# Дельта Берроуза
измеряет стилистическую близость между текстами, чтобы подтвердить авторство текстов

In [ ]:
import re
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка файла
input_file = "/content/Андреев. Рассказы. Сборник.txt"

print("=" * 50)
print("Загружаю файл...")
print("=" * 50)

with open(input_file, 'r', encoding='utf-8') as f:
    text = f.read()

# Разделяем по маркеру
marker = "===STORY==="
raw_stories = text.split(marker)
raw_stories = [s.strip() for s in raw_stories if s.strip()]

print(f"Найдено рассказов: {len(raw_stories)}")

# Извлекаем названия и тексты
story_names = []
story_texts = []

for story in raw_stories:
    lines = story.strip().split('\n')
    # Первая непустая строка = название
    name = ""
    text_lines = []
    for line in lines:
        if not name and line.strip():
            name = line.strip()
        elif name:
            text_lines.append(line)
    story_names.append(name)
    story_texts.append(' '.join(text_lines))

print("Рассказы:")
for i, name in enumerate(story_names):
    print(f"  {i+1}. {name}")

# Функция очистки
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яёa-z\s]', ' ', text)
    words = text.split()
    return words

# Функция расчета дельты
def get_freq(words):
    total = len(words)
    c = Counter(words)
    return {w: c[w]/total for w in c}

def burrows_delta(w1, w2, n=150):
    f1 = get_freq(w1)
    f2 = get_freq(w2)
    all_w = w1 + w2
    top = [w for w, _ in Counter(all_w).most_common(n)]

    v1 = np.array([f1.get(w, 0) for w in top])
    v2 = np.array([f2.get(w, 0) for w in top])

    mean = (v1 + v2) / 2
    std = np.std([v1, v2], axis=0)
    std[std == 0] = 1.0

    z1 = (v1 - mean) / std
    z2 = (v2 - mean) / std

    delta = np.mean(np.abs(z1 - z2))
    return delta

# Очищаем все тексты
print("\nОчищаю тексты...")
clean_stories = [clean_text(t) for t in story_texts]
print("Готово!")

# Строим матрицу
print("\nСчитаю матрицу дельты (это может занять пару минут)...")
n = len(story_names)
matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i == j:
            matrix[i][j] = 0.0
        elif i < j:
            d = burrows_delta(clean_stories[i], clean_stories[j])
            matrix[i][j] = d
            matrix[j][i] = d
        print(f"  Прогресс: {i+1}/{n} x {j+1}/{n}", end='\r')

print("\nГотово!")

# Визуализация
plt.figure(figsize=(12, 10))
sns.heatmap(matrix,
            xticklabels=story_names,
            yticklabels=story_names,
            annot=True,
            fmt='.2f',
            cmap='RdYlGn_r',
            vmin=0,
            vmax=np.max(matrix),
            cbar_kws={'label': 'Дельта Берроуза'},
            linewidths=0.5)
plt.title('Матрица дельты Берроуза: попарное сравнение рассказов Андреева', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('delta_matrix.png', dpi=150, bbox_inches='tight')
print("\nТепловая карта сохранена: delta_matrix.png")
plt.show()

# Статистика
print("\n" + "=" * 50)
print("РЕЗУЛЬТАТЫ:")
print("=" * 50)

# Находим самые похожие и самые разные пары
pairs = []
for i in range(n):
    for j in range(i+1, n):
        pairs.append((matrix[i][j], story_names[i], story_names[j]))

pairs.sort()

print("\nТОП-5 САМЫХ ПОХОЖИХ ПАР (минимальная дельта):")
for d, n1, n2 in pairs[:5]:
    print(f"  Δ = {d:.4f} | {n1}  ↔  {n2}")

print("\nТОП-5 САМЫХ РАЗНЫХ ПАР (максимальная дельта):")
for d, n1, n2 in pairs[-5:]:
    print(f"  Δ = {d:.4f} | {n1}  ↔  {n2}")

print("\nГотово!")

In [ ]:
import re
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка файла
input_file = "/content/Андреев. Рассказы. Сборник.txt"

print("=" * 50)
print("Загружаю файл...")
print("=" * 50)

with open(input_file, 'r', encoding='utf-8') as f:
    text = f.read()

# Разделяем по маркеру
marker = "===STORY==="
raw_stories = text.split(marker)
raw_stories = [s.strip() for s in raw_stories if s.strip()]

print(f"Найдено рассказов: {len(raw_stories)}")

# Извлекаем названия и тексты
story_names = []
story_texts = []

for story in raw_stories:
    lines = story.strip().split('\n')
    name = ""
    text_lines = []
    for line in lines:
        if not name and line.strip():
            name = line.strip()
        elif name:
            text_lines.append(line)
    story_names.append(name)
    story_texts.append(' '.join(text_lines))

# Функция очистки
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яёa-z\s]', ' ', text)
    words = text.split()
    return words

# Функция расчета дельты
def get_freq(words):
    total = len(words)
    if total == 0:
        return {}
    c = Counter(words)
    return {w: c[w]/total for w in c}

def burrows_delta(w1, w2, n=100):
    if len(w1) == 0 or len(w2) == 0:
        return 0.0
    f1 = get_freq(w1)
    f2 = get_freq(w2)
    all_w = w1 + w2
    top = [w for w, _ in Counter(all_w).most_common(n)]

    v1 = np.array([f1.get(w, 0) for w in top])
    v2 = np.array([f2.get(w, 0) for w in top])

    mean = (v1 + v2) / 2
    std = np.std([v1, v2], axis=0)
    std[std == 0] = 1.0

    z1 = (v1 - mean) / std
    z2 = (v2 - mean) / std

    delta = np.mean(np.abs(z1 - z2))
    return delta

CHUNK_SIZE = 300
MIN_CHUNKS = 5


print(f"\nРазмер кусочка: {CHUNK_SIZE} слов")
print(f"Минимум кусочков для карты: {MIN_CHUNKS}")
print("\nСтрою карты гомогенности...\n")

for idx, (name, raw_text) in enumerate(zip(story_names, story_texts)):
    # Очищаем текст
    words = clean_text(raw_text)
    total_words = len(words)


    chunks = []
    for i in range(0, total_words, CHUNK_SIZE):
        chunk = words[i:i+CHUNK_SIZE]
        if len(chunk) >= 50:
            chunks.append(chunk)

    n_chunks = len(chunks)

    print(f"{idx+1}. {name}")
    print(f"   Слов: {total_words} | Кусочков: {n_chunks}")


    if n_chunks < MIN_CHUNKS:
        print(f"Слишком мало кусочков, пропускаю\n")
        continue

    # Строим матрицу гомогенности
    matrix = np.zeros((n_chunks, n_chunks))

    for i in range(n_chunks):
        for j in range(n_chunks):
            if i == j:
                matrix[i][j] = 0.0
            elif i < j:
                d = burrows_delta(chunks[i], chunks[j])
                matrix[i][j] = d
                matrix[j][i] = d

    # Считаем среднюю дельту по рассказу
    upper_triangle = []
    for i in range(n_chunks):
        for j in range(i+1, n_chunks):
            upper_triangle.append(matrix[i][j])

    mean_delta = np.mean(upper_triangle)
    std_delta = np.std(upper_triangle)

    print(f"   Средняя дельта: {mean_delta:.4f} ± {std_delta:.4f}")

    # Интерпретация
    if mean_delta < 0.8:
        print(f"Рассказ ОЧЕНЬ однородный (стиль стабилен)")
    elif mean_delta < 1.2:
        print(f"Рассказ однородный (небольшие колебания стиля)")
    elif mean_delta < 1.6:
        print(f"Рассказ умеренно неоднородный (заметные перепады стиля)")
    else:
        print(f"Рассказ НЕОДНОРОДНЫЙ (сильные стилистические сломы)")

    # Рисуем карту
    fig, ax = plt.subplots(figsize=(10, 8))

    short_name = name[:50] + "..." if len(name) > 50 else name

    sns.heatmap(matrix,
                annot=False,
                cmap='RdYlGn_r',
                vmin=0,
                vmax=np.max(matrix) if np.max(matrix) > 0 else 1,
                cbar_kws={'label': 'Дельта Берроуза'},
                linewidths=0,
                ax=ax,
                square=True)

    ax.set_title(f'Карта гомогенности: {short_name}\n(Средняя Δ = {mean_delta:.3f})', fontsize=12)
    ax.set_xlabel('Кусочек текста (от начала к концу)')
    ax.set_ylabel('Кусочек текста (от начала к концу)')

    plt.tight_layout()

print("=" * 50)
print("ИТОГОВАЯ СВОДКА")
print("=" * 50)

print("Зелёный = кусочки стилистически похожи (текст однородный)")
print("Красный = кусочки стилистически разные (текст неоднородный)")
print("Диагональ всегда зелёная (кусочек сравнивается сам с собой)")
print()

print("\nВсе карты построены!")